# Step 6 — Compile Paper Results Tables

**Before running:** Attach your `step45-outputs` Kaggle dataset containing these files:
- `run2_aasist_mp3_only_results.json`
- `run3_aasist_opus_only_results.json`
- `run4_aasist_all_codecs_results.json`
- `run6_aasist_aac_only_results(1).json`
- `run7_rawnet2_mp3_only_results.json`
- `run8_rawnet2_all_codecs_results.json`

Run 5 data is embedded directly (recovered from PDF).

In [ ]:
import json, os
import numpy as np
from pathlib import Path

# ── Find the input directory ──────────────────────────────────
INPUT_ROOT = Path('/kaggle/input')
OUT_DIR    = Path('/kaggle/working')

# Search all subdirs for our JSON files
def find_json(name):
    for p in INPUT_ROOT.rglob(name):
        return p
    return None

# ── Run 5 data embedded from PDF ─────────────────────────────
RUN5_DATA = {
    'run': 'run5_aasist_mp3_opus', 'model': 'AASIST',
    'codec_set': 'mp3_opus', 'best_dev_eer': 0.5506, 'best_epoch': 30,
    'asvspooof2019_eval': {'clean': {'overall_eer': 1.2777, 'per_system': {
        'A07':0.431,'A08':0.326,'A09':0.017,'A10':0.489,'A11':0.326,
        'A12':0.407,'A13':0.268,'A14':0.163,'A15':0.431,'A16':0.832,
        'A17':1.704,'A18':3.375,'A19':0.710}}},
    'wavefake': {
        'clean':   {'ljspeech_melgan':27.35,'ljspeech_melgan_large':28.15,'ljspeech_hifiGAN':49.45,'ljspeech_waveglow':33.85,'ljspeech_full_band_melgan':49.85,'ljspeech_multi_band_melgan':49.95,'ljspeech_parallel_wavegan':50.75},
        'mp3_32':  {'ljspeech_melgan':30.15,'ljspeech_melgan_large':30.60,'ljspeech_hifiGAN':50.30,'ljspeech_waveglow':36.15,'ljspeech_full_band_melgan':51.40,'ljspeech_multi_band_melgan':51.45,'ljspeech_parallel_wavegan':52.80},
        'mp3_64':  {'ljspeech_melgan':27.45,'ljspeech_melgan_large':27.65,'ljspeech_hifiGAN':49.60,'ljspeech_waveglow':34.60,'ljspeech_full_band_melgan':50.30,'ljspeech_multi_band_melgan':50.75,'ljspeech_parallel_wavegan':51.50},
        'mp3_128': {'ljspeech_melgan':27.15,'ljspeech_melgan_large':27.40,'ljspeech_hifiGAN':49.75,'ljspeech_waveglow':34.55,'ljspeech_full_band_melgan':50.35,'ljspeech_multi_band_melgan':50.85,'ljspeech_parallel_wavegan':51.60},
        'aac_64':  {'ljspeech_melgan':28.55,'ljspeech_melgan_large':29.05,'ljspeech_hifiGAN':49.80,'ljspeech_waveglow':34.80,'ljspeech_full_band_melgan':49.45,'ljspeech_multi_band_melgan':50.55,'ljspeech_parallel_wavegan':51.10},
        'aac_96':  {'ljspeech_melgan':28.40,'ljspeech_melgan_large':29.10,'ljspeech_hifiGAN':49.55,'ljspeech_waveglow':34.75,'ljspeech_full_band_melgan':49.45,'ljspeech_multi_band_melgan':49.80,'ljspeech_parallel_wavegan':51.05},
        'opus_32': {'ljspeech_melgan':28.80,'ljspeech_melgan_large':28.45,'ljspeech_hifiGAN':50.50,'ljspeech_waveglow':35.05,'ljspeech_full_band_melgan':51.45,'ljspeech_multi_band_melgan':51.45,'ljspeech_parallel_wavegan':52.50},
        'opus_64': {'ljspeech_melgan':27.60,'ljspeech_melgan_large':27.95,'ljspeech_hifiGAN':49.20,'ljspeech_waveglow':34.35,'ljspeech_full_band_melgan':50.15,'ljspeech_multi_band_melgan':49.80,'ljspeech_parallel_wavegan':50.75},
    }
}

# ── Run file definitions ──────────────────────────────────────
RUNS = {
    2: ('run2_aasist_mp3_only_results.json',        'AASIST / MP3 only',    'AASIST',  'mp3_only'),
    3: ('run3_aasist_opus_only_results.json',       'AASIST / Opus only',   'AASIST',  'opus_only'),
    4: ('run4_aasist_all_codecs_results.json',      'AASIST / All codecs',  'AASIST',  'all'),
    5: (None,                                       'AASIST / MP3+Opus',    'AASIST',  'mp3_opus'),
    6: ('run6_aasist_aac_only_results(1).json',     'AASIST / AAC only',    'AASIST',  'aac_only'),
    7: ('run7_rawnet2_mp3_only_results.json',       'RawNet2 / MP3 only',   'RawNet2', 'mp3_only'),
    8: ('run8_rawnet2_all_codecs_results.json',     'RawNet2 / All codecs', 'RawNet2', 'all'),
}

# ── Load ─────────────────────────────────────────────────────
data = {}
for run_id, (fname, label, model, codec) in RUNS.items():
    if run_id == 5:
        data[5] = RUN5_DATA
        print(f'Run 5 ({label}): loaded from embedded data')
        continue
    p = find_json(fname)
    if p:
        with open(p) as f:
            data[run_id] = json.load(f)
        print(f'Run {run_id} ({label}): ✓ {p}')
    else:
        print(f'Run {run_id} ({label}): ✗ NOT FOUND — check dataset attachment')

print(f'\nLoaded runs: {sorted(data.keys())}')

In [ ]:
# ── Helper functions ──────────────────────────────────────────
def asv_eer(d, cond='clean'):
    try:    return d['asvspooof2019_eval'][cond]['overall_eer']
    except: return None

def wf_mean(d, cond='clean'):
    try:
        vals = list(d['wavefake'][cond].values())
        return float(np.mean(vals))
    except: return None

def wf_voc(d, voc, cond='clean'):
    try:    return d['wavefake'][cond][voc]
    except: return None

def fmt(v, w=8):
    return f'{v:{w}.2f}' if v is not None else f'{"N/A":>{w}}'

available = sorted(data.keys())

ASV_CONDS = ['clean','mp3_64','mp3_128','aac_96','opus_64']
WF_CONDS  = ['clean','mp3_64','aac_96','opus_64']
CLABEL    = {'clean':'Clean','mp3_64':'MP3 64k','mp3_128':'MP3 128k',
             'aac_64':'AAC 64k','aac_96':'AAC 96k','opus_32':'Opus 32k','opus_64':'Opus 64k'}
VOCODERS  = ['ljspeech_melgan','ljspeech_melgan_large','ljspeech_hifiGAN',
             'ljspeech_waveglow','ljspeech_full_band_melgan',
             'ljspeech_multi_band_melgan','ljspeech_parallel_wavegan']
VOC_SHORT = {'ljspeech_melgan':'MelGAN','ljspeech_melgan_large':'MelGAN-L',
             'ljspeech_hifiGAN':'HiFi-GAN','ljspeech_waveglow':'WaveGlow',
             'ljspeech_full_band_melgan':'FB-MelGAN',
             'ljspeech_multi_band_melgan':'MB-MelGAN',
             'ljspeech_parallel_wavegan':'ParWaveGAN'}
SYSTEMS   = ['A07','A08','A09','A10','A11','A12','A13','A14','A15','A16','A17','A18','A19']

lines = []
def pr(s=''):
    print(s)
    lines.append(s)

print('✓ Helpers ready')

In [ ]:
# ── TABLE 1: ASVspoof 2019 EER ────────────────────────────────
pr('='*78)
pr('  TABLE 1: ASVspoof 2019 LA — Overall EER (%) per Training Strategy & Eval Condition')
pr('='*78)
hdr = f"  {'Strategy':<24}" + ''.join(f" {CLABEL[c]:>9}" for c in ASV_CONDS) + f" {'Mean':>9}"
pr(hdr)
pr('  '+'-'*24 + '  '+'-'*9 + ('   '+'-'*8)*len(ASV_CONDS))

for r in available:
    label = RUNS[r][1]
    eers  = []
    row   = f'  {label:<24}'
    for c in ASV_CONDS:
        v = asv_eer(data[r], c)
        row += f' {fmt(v,9)}'
        if v: eers.append(v)
    row += f' {fmt(np.mean(eers) if eers else None, 9)}'
    pr(row)
pr()

In [ ]:
# ── TABLE 2: WaveFake mean EER ────────────────────────────────
pr('='*78)
pr('  TABLE 2: WaveFake Cross-Dataset — Mean EER (%) Across 7 Vocoders')
pr('  (RawNet2 ~50% = chance level, AASIST shows partial generalisation)')
pr('='*78)
hdr = f"  {'Strategy':<24}" + ''.join(f" {CLABEL[c]:>9}" for c in WF_CONDS) + f" {'Mean':>9}"
pr(hdr)
pr('  '+'-'*24 + '  '+'-'*9 + ('   '+'-'*8)*len(WF_CONDS))

for r in available:
    label = RUNS[r][1]
    vals  = []
    row   = f'  {label:<24}'
    for c in WF_CONDS:
        v = wf_mean(data[r], c)
        row += f' {fmt(v,9)}'
        if v: vals.append(v)
    row += f' {fmt(np.mean(vals) if vals else None, 9)}'
    pr(row)
pr()

In [ ]:
# ── TABLE 3: WaveFake per-vocoder (AASIST, clean) ─────────────
pr('='*78)
pr('  TABLE 3: WaveFake Per-Vocoder EER (%) — AASIST Runs, Clean Eval Condition')
pr('='*78)

aasist_runs = [r for r in available if RUNS[r][2]=='AASIST']
hdr = f"  {'Vocoder':<13}" + ''.join(f" {RUNS[r][3]:>12}" for r in aasist_runs)
pr(hdr)
pr('  '+'-'*13 + ('  '+'-'*12)*len(aasist_runs))

for voc in VOCODERS:
    row = f'  {VOC_SHORT[voc]:<13}'
    for r in aasist_runs:
        v = wf_voc(data[r], voc, 'clean')
        row += f' {fmt(v,12)}'
    pr(row)
pr()

In [ ]:
# ── TABLE 4: Per-system ASVspoof EER (clean) ──────────────────
pr('='*78)
pr('  TABLE 4: Per-System EER (%) on ASVspoof 2019 — Clean Eval Condition')
pr('='*78)

hdr = f"  {'System':<8}" + ''.join(f" {RUNS[r][3]:>12}" for r in available)
pr(hdr)
pr('  '+'-'*8 + ('  '+'-'*12)*len(available))

for sys_id in SYSTEMS:
    row = f'  {sys_id:<8}'
    for r in available:
        try:
            v = data[r]['asvspooof2019_eval']['clean']['per_system'][sys_id]
            row += f' {v:>12.2f}'
        except:
            row += f' {"N/A":>12}'
    pr(row)
pr()

In [ ]:
# ── KEY FINDINGS ──────────────────────────────────────────────
pr('='*78)
pr('  KEY FINDINGS — paste into paper draft')
pr('='*78)

# F1: AASIST ASVspoof clean range
aasist_clean = {r: asv_eer(data[r],'clean') for r in available if RUNS[r][2]=='AASIST'}
best_r = min(aasist_clean, key=lambda r: aasist_clean[r] or 99)
pr(f"\n  F1 — AASIST ASVspoof 2019 clean EER (all runs within 1.2–1.6%):")
for r,v in sorted(aasist_clean.items()):
    pr(f"      Run {r} {RUNS[r][3]:12s}: {v:.4f}%")

# F2: RawNet2 vs AASIST
if 7 in data and 2 in data:
    r2 = asv_eer(data[7],'clean'); aa = asv_eer(data[2],'clean')
    pr(f"\n  F2 — RawNet2 vs AASIST (clean ASVspoof EER):")
    pr(f"      AASIST  (MP3-only): {aa:.2f}%")
    pr(f"      RawNet2 (MP3-only): {r2:.2f}%  →  AASIST better by {r2-aa:.2f} pp")

# F3: WaveFake cross-domain
pr(f"\n  F3 — WaveFake cross-dataset mean EER (clean):")
for r in available:
    v = wf_mean(data[r],'clean')
    pr(f"      Run {r} {RUNS[r][1]:24s}: {v:.2f}%")

# F4: Hardest system
pr(f"\n  F4 — Hardest TTS system: A18 (3–27% EER across all runs)")
pr(f"       Easiest system:       A09 (< 0.05% for AASIST)")

pr()

# ── Save outputs ──────────────────────────────────────────────
txt_path = OUT_DIR / 'paper_tables.txt'
txt_path.write_text('\n'.join(lines))
print(f'\n✓ Tables saved → {txt_path}')

compiled = {}
for r in available:
    compiled[f"run{r}_{RUNS[r][3]}"] = {
        'model': RUNS[r][2], 'codec_set': RUNS[r][3],
        'best_epoch': data[r].get('best_epoch'),
        'best_dev_eer': data[r].get('best_dev_eer'),
        'asv_clean_eer': asv_eer(data[r],'clean'),
        'wf_clean_mean': wf_mean(data[r],'clean'),
        'asv_per_condition': {c: asv_eer(data[r],c) for c in ASV_CONDS},
        'wf_per_condition_mean': {c: wf_mean(data[r],c) for c in WF_CONDS},
    }

json_path = OUT_DIR / 'compiled_results.json'
json_path.write_text(json.dumps(compiled, indent=2))
print(f'✓ Compiled JSON saved → {json_path}')
print('\nDone! Download paper_tables.txt and compiled_results.json from /kaggle/working/')